In [1]:
from pathlib import Path

ROOT = Path.cwd().parent
DATA = ROOT / "data"
RANDOM_STATE = 42

In [2]:
import pandas as pd
from sklearn.metrics import f1_score


def compute_f1_scores(y_true: pd.DataFrame, y_pred: pd.DataFrame) -> pd.Series:
    """Return the F1 of each label column and the macro F1 (their mean)."""
    scores = pd.Series(f1_score(y_true, y_pred, average=None), index=y_true.columns)
    scores["macro_f1"] = scores.mean()
    return scores

In [3]:
train = pd.read_csv(DATA / "train.csv")
label_columns = [column for column in train.columns if column != "image_id"]
y_true = train[label_columns]

named_pet_columns = ["Moris", "Motya", "Biatrix"]
predictions = {
    "all ones": pd.DataFrame(1, index=y_true.index, columns=label_columns),
    "perfect": y_true.copy(),
    "perfect Cat/Dog, zeros for named pets": y_true.assign(**{column: 0 for column in named_pet_columns}),
}

pd.DataFrame({name: compute_f1_scores(y_true, y_pred) for name, y_pred in predictions.items()}).T

,Cat,Dog,Moris,Motya,Biatrix,macro_f1
all ones,0.585684,0.581429,0.009094,0.006828,0.006666,0.23794
perfect,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000
"perfect Cat/Dog, zeros for named pets",1.000000,1.000000,0.000000,0.000000,0.000000,0.40000


## Conclusions

- The two perfect cases give trivial scores. The first copies the true labels and scores 1.0 on every label. The second scores 1.0 on Cat and Dog and 0.0 on the 3 named pets, so macro F1 is 0.4 (2 of 5 labels). Both match a hand calculation, so the function is correct.
- The all-ones case is a silly baseline. It scores about 0.58 on Cat and Dog and near 0 on the named pets, so macro F1 is only about 0.24.
- Plain accuracy or a micro average would hide this weakness, because Cat and Dog dominate the data. Macro F1 gives each label 20%, so the 3 rare named pets cannot be ignored. We need macro F1.